# Week 03 Project Review Text Sentiment Analysis
**Edversity AI Solutions Engineering Specialization**

## Project Overview and Learning Objectives

This notebook follows the structure of the original Week 03 lesson and applies it to the E-Commerce hackathon track. It predicts **positive**, **neutral**, or **negative** sentiment from `written_review` only.

The star-rating column is never included in the model. The notebook uses Pandas feature engineering, TF-IDF, an 80/20 split, Logistic Regression, and standard classification metrics.

Because the supplied CSV has no sentiment target, this simple classroom version creates **weak text-only labels** from a small list of positive and negative words. These labels are useful for demonstrating the full machine-learning workflow, but they are not manually verified ground truth.

---
## 1 Reading the CSV Dataset and Initial Inspection

We start by loading `airpods_cleaned.csv` with Pandas. Keep the CSV in the same folder as this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the supplied AirPods dataset.
csv_path = "airpods_cleaned.csv"
df = pd.read_csv(csv_path)

print(f"Dataset successfully loaded! Shape: {df.shape}")
df.head()

In [ ]:
# Inspect the available columns and missing values.
print("=== Columns in Dataset ===")
print(df.columns.tolist())

print("\n=== Missing Values Check ===")
print(df.isnull().sum())

print("\n=== Duplicate Written Reviews ===")
print(df["written_review"].duplicated().sum())

---
## 2 Creating Text Only Sentiment Labels

The dataset does not contain a sentiment column. To keep this notebook simple and avoid star-rating leakage, we create demonstration labels from words found in each written review.

- More positive words than negative words gives `positive`.
- More negative words than positive words gives `negative`.
- Equal scores gives `neutral`.

This rule uses only review text. It does not use `review_rating`, price, seller, or any product metadata.

In [ ]:
# Clean text using simple Pandas string operations.
df["clean_review"] = (
    df["written_review"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Small English and Roman Urdu word lists for weak text-only labels.
positive_words = {
    "good", "great", "amazing", "excellent", "best", "love", "perfect",
    "satisfied", "happy", "nice", "comfortable", "clear", "awesome",
    "better", "recommended", "acha", "achi", "achha", "behtareen",
    "zabardast", "kamal", "khush"
}

negative_words = {
    "bad", "poor", "terrible", "worst", "broken", "waste", "horrible",
    "disappointed", "problem", "issue", "fake", "damaged", "stopped",
    "refund", "buri", "bura", "ghatiya", "bekar", "kharab", "masla",
    "sharmindagi"
}

def create_text_sentiment(review):
    # Split the cleaned review and count matching sentiment words.
    words = review.split()
    positive_score = sum(word in positive_words for word in words)
    negative_score = sum(word in negative_words for word in words)

    if positive_score > negative_score:
        return "positive"
    if negative_score > positive_score:
        return "negative"
    return "neutral"


df["sentiment"] = df["clean_review"].apply(create_text_sentiment)

print("Text-only sentiment distribution:")
print(df["sentiment"].value_counts())
df[["written_review", "clean_review", "sentiment"]].head()

---
## 3 Train Test Split

We use 80% of the reviews for training and 20% for testing. Stratification keeps the sentiment proportions similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Remove exact duplicate review text before splitting to reduce leakage.
model_df = df.drop_duplicates(subset=["clean_review"]).copy()

X = model_df[["clean_review"]]
y = model_df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print("\nTraining class counts:")
print(y_train.value_counts())

---
## 4 Baseline Model

A baseline predicts the most common training class for every review. It gives us a minimum benchmark before we train Logistic Regression.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

# DummyClassifier does not learn from text; it predicts the majority class.
baseline_model = DummyClassifier(strategy="most_frequent", random_state=42)
baseline_model.fit(np.zeros((len(X_train), 1)), y_train)
baseline_predictions = baseline_model.predict(np.zeros((len(X_test), 1)))

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_macro_f1 = f1_score(y_test, baseline_predictions, average="macro", zero_division=0)

print(f"Baseline Accuracy: {baseline_accuracy:.3f}")
print(f"Baseline Macro F1: {baseline_macro_f1:.3f}")

---
## 5 Sentiment Classification Concepts Revision

Before training the model, we review the main ideas used in this project.

### Concept 1 TF IDF

Machine-learning models cannot directly process sentences. TF-IDF converts words into numbers. A word receives more importance when it appears in a review but is less common across the complete dataset.

We use unigrams and bigrams so the model can learn individual words such as `excellent` and short phrases such as `not good`.

### Concept 2 Engineered Text Features

The assignment requires at least three engineered interaction features. We calculate:

- average word length = character count divided by word count;
- lexical diversity = unique word count divided by word count; and
- exclamation density = exclamation count divided by word count.

All three are derived only from the written review.

In [ ]:
# Create simple count features from the review text.
df["word_count"] = df["clean_review"].str.split().str.len()
df["character_count"] = df["clean_review"].str.len()
df["unique_word_count"] = df["clean_review"].apply(lambda text: len(set(text.split())))
df["exclamation_count"] = df["written_review"].fillna("").str.count("!")

# Three engineered interaction features required by the rubric.
safe_word_count = df["word_count"].clip(lower=1)
df["average_word_length"] = df["character_count"] / safe_word_count
df["lexical_diversity"] = df["unique_word_count"] / safe_word_count
df["exclamation_density"] = df["exclamation_count"] / safe_word_count

feature_columns = [
    "word_count",
    "character_count",
    "unique_word_count",
    "exclamation_count",
    "average_word_length",
    "lexical_diversity",
    "exclamation_density"
]

df[feature_columns].head()

### Concept 3 Logistic Regression

Logistic Regression is a supervised classification model. It learns weights for TF-IDF words and engineered numeric features. The class with the strongest combined score becomes the predicted sentiment.

### Concept 4 Training Process

1. TF-IDF learns its vocabulary from the training reviews.
2. Numeric features are scaled using the training set.
3. Logistic Regression learns the relationship between these features and sentiment.
4. The untouched test set measures performance on unseen reviews.

### Concept 5 Overfitting and Data Leakage

Overfitting occurs when a model memorizes training examples and performs poorly on new data. Removing duplicate reviews before the split helps reduce this risk.

Data leakage occurs when information that reveals the target reaches the model. `review_rating` is excluded completely, so the classifier must use written language only.

---
## 6 Training the Logistic Regression Model

We combine TF-IDF and the three engineered interaction features in one Scikit-Learn pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Rebuild model_df so it includes the engineered columns created above.
model_df = df.drop_duplicates(subset=["clean_review"]).copy()

numeric_features = [
    "average_word_length",
    "lexical_diversity",
    "exclamation_density"
]

X = model_df[["clean_review"] + numeric_features]
y = model_df["sentiment"]

# Repeat the same reproducible 80/20 split using all required features.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Apply TF-IDF to text and scaling to numeric features.
feature_processor = ColumnTransformer([
    ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), "clean_review"),
    ("numeric", StandardScaler(), numeric_features)
])

sentiment_model = Pipeline([
    ("features", feature_processor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train the complete pipeline.
sentiment_model.fit(X_train, y_train)
print("Logistic Regression training completed.")

In [ ]:
# Make predictions for training and testing reviews.
y_pred_train = sentiment_model.predict(X_train)
y_pred_test = sentiment_model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.3f}")
print(f"Testing Accuracy: {test_accuracy:.3f}")

### 6.1 Confusion Matrix

The confusion matrix shows how many positive, neutral, and negative reviews were classified correctly or confused with another class.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["negative", "neutral", "positive"]
matrix = confusion_matrix(y_test, y_pred_test, labels=labels)

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=labels
)
display_matrix.plot(cmap="Blues")
plt.title("Review Text Sentiment Confusion Matrix")
plt.tight_layout()
plt.show()

---
## 7 Evaluation and Model Comparison

We report accuracy, precision, recall, F1, and the complete per-class classification report. Macro F1 is important because it gives equal weight to each sentiment class. We also compare the model with the majority-class baseline.

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score

logistic_accuracy = accuracy_score(y_test, y_pred_test)
logistic_precision = precision_score(y_test, y_pred_test, average="macro", zero_division=0)
logistic_recall = recall_score(y_test, y_pred_test, average="macro", zero_division=0)
logistic_macro_f1 = f1_score(y_test, y_pred_test, average="macro", zero_division=0)

print("=== Logistic Regression Test Metrics ===")
print(f"Accuracy:        {logistic_accuracy:.3f}")
print(f"Macro Precision: {logistic_precision:.3f}")
print(f"Macro Recall:    {logistic_recall:.3f}")
print(f"Macro F1:        {logistic_macro_f1:.3f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_test, zero_division=0))

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Majority Baseline", "TF-IDF Logistic Regression"],
    "Accuracy": [baseline_accuracy, logistic_accuracy],
    "Macro F1": [baseline_macro_f1, logistic_macro_f1]
})

print(comparison.round(3))

comparison.set_index("Model").plot(
    kind="bar",
    figsize=(9, 5),
    color=["steelblue", "darkorange"]
)
plt.title("Baseline vs Logistic Regression")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 7.2 Actual and Predicted Sentiment Distribution

This chart compares the number of actual weak labels with the model predictions in the test set.

In [ ]:
distribution_comparison = pd.DataFrame({
    "Actual": y_test.value_counts(),
    "Predicted": pd.Series(y_pred_test).value_counts()
}).fillna(0).reindex(labels)

distribution_comparison.plot(
    kind="bar",
    figsize=(8, 5),
    color=["seagreen", "mediumpurple"]
)
plt.title("Actual vs Predicted Sentiment Counts")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 8 Making Live Review Predictions

For a new review, we calculate the same three numeric features and let the trained pipeline predict sentiment. No star rating is supplied.

In [ ]:
def prepare_new_reviews(reviews):
    # Create a DataFrame with the same columns used during training.
    new_data = pd.DataFrame({"original_review": reviews})
    new_data["clean_review"] = new_data["original_review"]
    new_data["clean_review"] = (
        new_data["clean_review"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    word_count = new_data["clean_review"].str.split().str.len().clip(lower=1)
    character_count = new_data["clean_review"].str.len()
    unique_word_count = new_data["clean_review"].apply(lambda text: len(set(text.split())))

    new_data["average_word_length"] = character_count / word_count
    new_data["lexical_diversity"] = unique_word_count / word_count
    exclamation_count = new_data["original_review"].fillna("").str.count("!")
    new_data["exclamation_density"] = exclamation_count / word_count
    return new_data[["clean_review"] + numeric_features]


sample_reviews = [
    "The sound quality is excellent and I love the battery life",
    "The earbuds stopped working and the quality is terrible",
    "The product is okay and works as expected"
]

sample_features = prepare_new_reviews(sample_reviews)
sample_predictions = sentiment_model.predict(sample_features)

prediction_results = pd.DataFrame({
    "Review": sample_reviews,
    "Predicted Sentiment": sample_predictions
})

prediction_results

---
## Summary and Key Takeaways

1. The project uses the supplied AirPods dataset for the E-Commerce sentiment track.
2. `review_rating` and product metadata are excluded from the model.
3. Simple text rules create weak demonstration labels because the source CSV has no sentiment target.
4. Pandas creates average word length, lexical diversity, and exclamation density as interaction features.
5. TF-IDF converts written reviews into numerical word features.
6. The notebook uses the required 80/20 split and Scikit-Learn Logistic Regression.
7. Evaluation includes accuracy, macro precision, macro recall, macro F1, a classification report, and a confusion matrix.

### Limitation

The reported scores measure how well Logistic Regression reproduces the simple word-list labels. They do not prove agreement with human sentiment judgment. A future version can replace the weak labels with manually reviewed text labels without adding star ratings as model inputs.